In [1]:
import pandas as pd
import scipy as sc
import numpy as np
import matplotlib.pyplot as plt
from docx import Document

In [2]:
#Define file path to read in experimental data
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()  # Hide the main window

full_file_path = filedialog.askopenfilename()

from pathlib import Path
filename = Path(f'{full_file_path}').name
directory = full_file_path.replace(filename, "")
filename = filename.replace('.xlsx', "")
workbook = pd.ExcelFile(full_file_path)

In [3]:
#Fix Column labels and generate control averages
def control_averages(df):
    dmso_arr = np.concatenate((df.iloc[0:8, 0], df.iloc[0:8, 12], df.iloc[8:16,11], df.iloc[8:16,23]))
    grp_arr = np.concatenate((df.iloc[0:8, 11], df.iloc[0:8, 23], df.iloc[8:16,0], df.iloc[8:16,12]))
    dmso_nan_mask = np.isnan(dmso_arr)
    grp_nan_mask = np.isnan(grp_arr)
    dmso_filtered = dmso_arr[~dmso_nan_mask]
    grp_filtered = grp_arr[~grp_nan_mask]
    dmso_avg = np.mean(dmso_filtered)
    grp_avg = np.mean(grp_filtered)
    
    return dmso_avg, grp_avg

In [4]:
#Remove control wells to isolate test samples, then normalize data using controls
def normalize_data(df, concdict, neg_avg, pos_avg):
    
    #Account for partial plate by removing empty rows, assuming empty rows are at bottom of plate
    concdict = {k: v for k,v in concdict.items() if k==k}
    num_cmpds = len(concdict.keys())
    
    compddf = df.drop([0, 11, 12, 23], axis=1)
    compddf = compddf[0:num_cmpds]
    compddf = compddf.set_axis(range(20), axis = 1)
    
    #Perform Normalization
    norm_compddf = pd.DataFrame(index=range(num_cmpds), columns=range(20))
    for a in range(num_cmpds):
        for b in range(20):
            norm_compddf.iloc[a,b] = (compddf.iloc[a,b] - neg_avg)/(pos_avg - neg_avg)*100
        
    norm_compddf.index = concdict.keys()
    norm_compddict = dict(zip(concdict.keys(), norm_compddf.values))

    return norm_compddict, concdict, num_cmpds

In [5]:
#Generate full lists of concentrations for all compounds

def dilutioncalc(start_conc):
    dilutions = np.zeros(10)
    dilutions[0] = start_conc
    for i in range(9):
        dilutions[i+1] = dilutions[i]/4
    M_dilutions = dilutions*1e-6
    log_dilutions = np.log10(M_dilutions)
    log_dilutions = np.tile(log_dilutions, 2)
    return np.array(log_dilutions)

In [6]:
#Load in curve fit and use four parameter logistic fits to then calculate curve fits

from scipy.optimize import curve_fit

# Four Parameter Logistic (4PL) function
def four_param_logistic(X, A, B, C, D):
    return D + (A - D) / (1 + 10**((C-X)*B))

def fit_4PLs(Xs_dict, Ys_dict, guess, bounds):
    param_sets = {}
    for sample_key in Xs_dict:
        Xs = np.array(Xs_dict[sample_key], dtype=np.float64)
        Ys = np.array(Ys_dict[sample_key], dtype=np.float64)

        # Check for equal lengths
        if len(Xs) != len(Ys):
            print(f"[⚠️] Length mismatch for {sample_key}: Xs={len(Xs)}, Ys={len(Ys)}")
            param_sets[sample_key] = [np.nan] * 4
            continue

        # Try fitting
        try:
            params, _ = curve_fit(four_param_logistic, Xs, Ys, p0=guess, bounds=bounds, maxfev=10000, nan_policy = 'omit')
            param_sets[sample_key] = params
        except RuntimeError as e:
            print(f"[❌] Fit failed for {sample_key}: {str(e)}")
            param_sets[sample_key] = [np.nan] * 4
        except Exception as e:
            print(f"[❌] Unexpected error for {sample_key}: {str(e)}")
            param_sets[sample_key] = [np.nan] * 4

    return param_sets   

In [7]:
#Let's Plot!

def Dose_Response_Plot(param_sets, concdict, dilndict, directory, filename, analysis_sheet, num_cmpds):
    #Prepare x parameters
    lower_x_key = min(concdict, key=concdict.get)
    lower_x = min(dilndict[lower_x_key])
    upper_x_key =  max(concdict, key=concdict.get)
    upper_x = max(dilndict[upper_x_key])
    x_fit = np.linspace(lower_x, upper_x, 500)
    
    #Generate Document to Upload figure files to
    doc = Document()
    doc.add_heading(f'{filename} {analysis_sheet} Results', 0)
    
    #Make table with 3 columns for Name, EC50, %Max
    tbl_length = num_cmpds 
    tbl = doc.add_table(tbl_length+1, 5)
    tbl.style = 'Light Grid Accent 1'
    tbl.cell(0,0).text = 'Compound'
    tbl.cell(0,1).text = 'EC50 (M)'
    tbl.cell(0,2).text = '% Max'
    tbl.cell(0,3).text = 'Dotmatics EC50 (M)'
    tbl.cell(0,4).text = 'Dotmatics % Max'
    compd_list = list(param_sets.keys())
    
    for compound, pset in param_sets.items():
        compd_index = compd_list.index(compound) 
        tbl.cell(compd_index+1, 0).text = str(compound)
        tbl.cell(compd_index+1, 1).text = f"{10**pset[2]:.2e}"
        tbl.cell(compd_index+1, 2).text = f"{pset[0]:.1f}"
    doc.add_page_break()
    
    for compound, pset in param_sets.items(): 
        y_fit = four_param_logistic(x_fit, *pset)
        plt.figure(figsize=(5, 3.75))
        plt.scatter(dilndict[compound], norm_compddict[compound], color='blue', label='Raw data', alpha=0.6)
        
        # #Optionally plot the mean at each unique X
        #unique_Xs = np.unique(dilndict[compound])
        #means = [np.mean(norm_compddict[compound][dilndict[compound] == x]) for x in unique_Xs]
        #plt.scatter(unique_Xs, means, color='black', label='Mean per concentration', marker='o', zorder=3)
        
        plt.plot(x_fit, y_fit, color='red', label='4PL fit', linewidth=2)
        plt.xlabel('Log10(Concentration)')
        plt.ylabel('Response')
        plt.title(f'4-Parameter Logistic Fit of {compound}')
        plt.legend()
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(f'{compound}.png')
        doc.add_picture(f'{compound}.png')
        plt.close()
    
    #Finally, save document
    doc.save(f'{directory}{filename} {analysis_sheet}.docx')

In [8]:
#Get format of Excel tab names to read through in loop for multiple plate analyses
analysis_format = 'Microplate Analysis Plate '
plate_layout_format = ['Plate', 'Layout']

num_plates = int(input('Please enter the number of plates to analyze: '))

for i in range(1,num_plates+1):
    
    #Take relevant plate data in as strings to check for outliers which are marked with asterisks or blank
    analysis_sheet_raw = f'{analysis_format}{i}'
    analysis_sheet = next((s for s in workbook.sheet_names if s.strip() == analysis_sheet_raw), None)
    
    #Get start concentrations
    plate_layout_sheet_raw = f'{plate_layout_format[0]} {i} {plate_layout_format[1]}'
    plate_layout_sheet = next((s for s in workbook.sheet_names if s.strip() == plate_layout_sheet_raw), None)
    
    if not analysis_sheet or not plate_layout_sheet:
        print(f"Warning: Could not find matching sheet for Plate {i}. Skipping.")
        continue

    #Read in analysis tab data as strings in case of outliers
    df = pd.read_excel(workbook, sheet_name=analysis_sheet, skiprows=3, nrows = 16, usecols='D:AA', dtype=str)
    # Convert to numeric, coercing errors from outliers to NaN
    df_numeric = df.apply(pd.to_numeric, errors='coerce')
    df = df_numeric.set_axis(range(24), axis = 1)

    #Read in compound layout data
    concdf = pd.read_excel(workbook, sheet_name=plate_layout_sheet, skiprows=22, nrows = 17, usecols='C:E', names = ['Name', 'Stock', 'Top Conc'])
    concdict = dict(zip(concdf['Name'], concdf['Top Conc']))
    
    #Invoke control_averages function 
    dmso_avg, grp_avg  = control_averages(df)

    #Invoke normalize_data function
    norm_compddict, concdict, num_cmpds = normalize_data(df, concdict, dmso_avg, grp_avg)

    #Invoke dilutioncalc 
    dilndict = {i: dilutioncalc(j) for (i,j) in concdict.items()}

    #Prepare fit parameters and invoke fit_4PLs
    guess = [1, 1, -9, 1]
    bounds = [[0, 0, -13, -50], [200, 10, -4, 100]]
    param_sets = fit_4PLs(dilndict, norm_compddict,guess, bounds)

    Dose_Response_Plot(param_sets, concdict, dilndict, directory, filename, analysis_sheet, num_cmpds)

    print(f'Plate {i} complete!')

print('Have a great day!')

Please enter the number of plates to analyze:  4


Plate 1 complete!
Plate 2 complete!
Plate 3 complete!
Plate 4 complete!
Have a great day!
